# Word Flashcard Vocab Learner: demo

This notebook walks through the package one step at a time, with the same functions the
`word-flashcard` app uses.

- Everything is saved in a **temporary demo folder**, so your real deck
  (`~/.word_flashcard/deck.json`) is never touched.
- Only the cells under *needs internet* call the Free Dictionary API (no key needed).
  Without internet they use a saved example, so the rest of the notebook still runs.
- The AI tutor cells only call the LLM when `LLM_API_KEY` is set in your `.env`.
- Run the cells from top to bottom (**Run All** works too).

In [ ]:
import json
import os
import shutil
import tempfile
from datetime import date

import requests

import word_flashcard_vocab_learner as app
from word_flashcard_vocab_learner import (
    cardDeck,
    connectFreeDictionaryAPI as freeDict,
    exportDeckToCSV,
    genAITutor,
    importVocabulary,
    leitnerScheduler,
    removeWords,
    reviewCards,
    showProgress,
    terminalDisplay,
)
from word_flashcard_vocab_learner.flashCard import Card

# A fresh folder for this demo, so nothing here changes your real deck.
demoFolder = tempfile.mkdtemp(prefix="flashcard-demo-")
deckPath = os.path.join(demoFolder, "deck.json")
today = date.today()

print("Demo folder:", demoFolder)

## 1. Look up a word (needs internet)

`getWordDefinition` asks the Free Dictionary API about a word and returns a plain dict:
the spelling the dictionary matched, up to 3 definitions, and up to 2 example sentences
for each definition.

In [ ]:
try:
    result = freeDict.getWordDefinition("catastrophic")
except requests.RequestException as error:
    print(f"No internet ({error}), so a saved example is used.")
    result = {
        "word": "catastrophic",
        "definitions": [
            "Involving or causing sudden great damage or suffering.",
            "Extremely unfortunate or unsuccessful.",
        ],
        "examples": [["The storm had a catastrophic effect on the harbour."], []],
    }

result

`formatWordDefinition` turns that dict into the text the app prints. Long lines wrap to
fit the screen.

In [ ]:
print(freeDict.formatWordDefinition(result))

`sendWordDefinition` does both steps at once. The dictionary is case-sensitive, so a word
is tried as typed, then in lowercase, then capitalised (`london` finds `London`). A word
it does not know comes back with no definitions.

In [ ]:
for eachWord in ["motivation", "london", "qwzxv"]:
    try:
        print(freeDict.sendWordDefinition(eachWord))
    except requests.RequestException as error:
        print(f"{eachWord}: no internet ({error})")
    print()

## 2. Make flashcards

A `Card` holds the word, its meanings and examples, the Chinese translation, and its
spaced-repetition state: the Leitner box (1 to 5), the next review date and the score.
`Card.fromLookup` turns a dictionary result into a new card: box 1, due today.

In [ ]:
lookedUpCard = Card.fromLookup(result)
lookedUpCard

A card can also be written by hand. `toDict()` is exactly what is saved in `deck.json`.

In [ ]:
handMadeCards = [
    Card(
        "ambiguous",
        ["Open to more than one interpretation."],
        [["The ending of the film is ambiguous."]],
        "含糊不清",
    ),
    Card(
        "reluctant",
        ["Unwilling and hesitant."],
        [["She was reluctant to leave."]],
        "不情願",
    ),
    Card(
        "amble",
        ["Walk at a slow, relaxed pace."],
        [["We ambled along the beach."]],
        "漫步",
    ),
    Card(
        "ice cream",
        ["A sweet frozen food made from milk."],
        [["We had ice cream after dinner."]],
        "雪糕",
    ),
]

handMadeCards[0].toDict()

`Card.fromDict` reads a card back from `deck.json`. If someone edited the file by hand and
left a wrong value, the value is repaired instead of breaking the app: the box is kept
between 1 and 5, a counter that is not a number becomes 0, and a date that is not a date
becomes today.

In [ ]:
handEdited = {
    "word": "  Zebra ",
    "definitions": None,
    "box": "9",
    "correctCount": "-2",
    "nextReview": "soon",
}

Card.fromDict(handEdited)

## 3. Keep the cards in a deck

A deck is a list of cards. `cardDeck` saves it as JSON, loads it again, and finds words
whatever their upper / lower case. A deck file that does not exist yet loads as an empty
deck.

In [ ]:
cards = cardDeck.loadDeck(deckPath)
print("Cards in a new deck:", len(cards))

for eachCard in [lookedUpCard, *handMadeCards]:
    cardDeck.addCard(cards, eachCard)

# addCard refuses a word that is already in the deck, whatever its case.
print("Added AMBIGUOUS again?", cardDeck.addCard(cards, Card("AMBIGUOUS", ["unclear"])))

cardDeck.saveDeck(cards, deckPath)
print("Saved", terminalDisplay.plural(len(cards), "card"), "to", deckPath)

Loading the file gives the same cards back. `findCard` ignores case, and `sortByWord`
gives the alphabetical order that the deck screen and the remove screen number from.

In [ ]:
cards = cardDeck.loadDeck(deckPath)

print("Find 'Reluctant':", cardDeck.findCard(cards, "Reluctant").word)
print("Find 'zebra':", cardDeck.findCard(cards, "zebra"))
print("In order:", [eachCard.word for eachCard in cardDeck.sortByWord(cards)])

This is what the deck file looks like (the first card):

In [ ]:
with open(deckPath, encoding="utf-8") as deckFile:
    print(json.dumps(json.load(deckFile)[0], ensure_ascii=False, indent=2))

## 4. Spaced repetition (Leitner boxes)

Every card sits in a box from 1 to 5. A right answer moves it up one box, so it comes back
later; a wrong answer sends it back to box 1, so it comes back tomorrow.

In [ ]:
for eachBox, eachDays in leitnerScheduler.INTERVAL_DAYS.items():
    print(f"Box {eachBox}: next review after {terminalDisplay.plural(eachDays, 'day')}")

Here one card is answered on the days it is due. `recordAnswer` updates the card the same
way a review does.

In [ ]:
card = Card("practice", ["Repeated exercise to get better at something."])
day = today

for isCorrect in [True, True, False, True]:
    leitnerScheduler.recordAnswer(card, isCorrect, day)
    answer = "right" if isCorrect else "wrong"
    whenNext = leitnerScheduler.describeNextReview(card, day)
    print(f"{day}  {answer}  ->  box {card.box}, back {whenNext} ({card.nextReview})")
    day = date.fromisoformat(card.nextReview)

`getDueCards` picks the cards to review on a day, and `nextDueDate` says when the next one
is due. A new card is due straight away.

In [ ]:
print("Due today:", [eachCard.word for eachCard in cardDeck.getDueCards(cards, today)])

# Answer every card right today: then nothing is due until the next review date.
for eachCard in cards:
    leitnerScheduler.recordAnswer(eachCard, True, today)

print("Due today after reviewing:", cardDeck.getDueCards(cards, today))
print("Next review:", cardDeck.nextDueDate(cards))

## 5. How answers are checked

The review modes show clues with the word hidden, then compare what you typed (or said)
with the word. These are the pieces the review screen is built from.

In [ ]:
print(reviewCards.formatClues(cardDeck.findCard(cards, "ambiguous")))

In [ ]:
print(reviewCards.maskWord("I have two cats, but no category.", "cat"))
print(reviewCards.makeHint("ice cream"))

# A typed answer ignores case, extra spaces, curly quotes and a full stop at the end.
print(reviewCards.isSpellingCorrect("  Ambiguous. ", "ambiguous"))
print(reviewCards.isSpellingCorrect("ambigous", "ambiguous"))

# A spoken answer counts when the word is said, even inside a short phrase.
print(reviewCards.isSpokenCorrect("(clears throat) It's ambiguous!", "ambiguous"))

A whole review session asks for answers with `input()`. To try one here, set
`RUN_INTERACTIVE = True` and run the cell (type `q` to stop). This is extra practice, so
it does not change the boxes. You can also run `word-flashcard` in a terminal.

In [ ]:
RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    summary = reviewCards.reviewCards(
        cards, reviewCards.MODE_SPELLING, updateSchedule=False
    )
    print(summary)

## 6. Find vocabulary in a text

`extractWords` is what the import screen uses. A **word list** gives the word at the start
of each line (numbers, meanings and title lines are left out); a **passage** gives its
less common words, without very common words and names.

In [ ]:
wordList = """Unit 3 vocabulary
1. ambiguous - unclear
2. reluctant: not willing
3) ice cream (n.) 雪糕
- persevere 堅持
"""

importVocabulary.extractWords(wordList)

In [ ]:
passage = (
    "Last week Mary was reluctant to answer, because the question was ambiguous. "
    "She ambled home and thought about it for hours. The answer wasn't obvious, "
    "but she didn't give up: she persevered."
)

# "Mary" is a name, and words like "week", "answer" or "thought" are too common.
importVocabulary.extractWords(passage)

Words already in the deck are left out, and `parseSelection` reads the numbers typed on
the import screen: Enter = all, `1,3-4` = some, a number that is not on the list = `None`.

In [ ]:
foundWords = importVocabulary.extractWords(wordList)
print("New words:", importVocabulary.removeKnownWords(foundWords, cards))

print(importVocabulary.parseSelection("1,3-4", 4))
print(importVocabulary.parseSelection("", 4))
print(importVocabulary.parseSelection("9", 4))

A CSV file is read by its first column. When it has a Chinese column (like the file
exported in step 8), those translations are kept too.

In [ ]:
csvPath = os.path.join(demoFolder, "words.csv")
with open(csvPath, "w", encoding="utf-8") as csvFile:
    csvFile.write("Word,Chinese\npersevere,堅持\nobvious,明顯\n")

print(importVocabulary.readTextFromFile(csvPath))
print(importVocabulary.readCSVTranslations(csvPath, ","))

## 7. Progress

`showProgress` prints the progress screen. A few answers are made up first, so there is
something to show.

In [ ]:
madeUpAnswers = [[True, True, True], [False, True], [False, False, True], [True], []]

for eachCard, eachAnswers in zip(cards, madeUpAnswers):
    for isCorrect in eachAnswers:
        leitnerScheduler.recordAnswer(eachCard, isCorrect, today)

showProgress.showProgress(cards)

## 8. Export to CSV

`exportDeckToCSV` writes a file that opens in Excel or Google Sheets (Chinese included)
and can be imported into Quizlet or Anki. A cell that a spreadsheet would run as a formula
gets a leading `'`.

In [ ]:
exportPath = os.path.join(demoFolder, "vocabulary.csv")
count = exportDeckToCSV.exportDeckToCSV(cards, exportPath)

print(f"Exported {terminalDisplay.plural(count, 'word')}:")
print()
with open(exportPath, encoding="utf-8-sig") as exportFile:
    print(exportFile.read())

## 9. Choose words to remove

The remove screen understands words, numbers, ranges, `*` patterns and boxes.
`selectCards` only works out which cards are meant; nothing is removed here.

In [ ]:
numberedCards = cardDeck.sortByWord(cards)

for index, eachCard in enumerate(numberedCards, start=1):
    print(f"{index}. {eachCard.word} (box {eachCard.box})")
print()

for eachAnswer in ["amb*", "1, 3-4", "box 2", "zebra"]:
    selected, problems = removeWords.selectCards(eachAnswer, numberedCards)
    words = [eachCard.word for eachCard in selected]
    print(f"{eachAnswer!r:9} -> {words} {' '.join(problems)}")

## 10. AI tutor (optional, needs `LLM_API_KEY`)

With an LLM key in `.env`, the app adds Chinese translations, gives feedback in Sentence
mode, and can pick the most useful words from a passage. Translations are cached, so each
word is only sent once (here the cache is kept in the demo folder).

In [ ]:
if genAITutor.isAIAvailable():
    print(genAITutor.translateWord("persevere", "Keep trying.", cacheDir=demoFolder))
    print(genAITutor.checkSentence("reluctant", "Unwilling.", "I am reluctant of go."))
else:
    print("LLM_API_KEY is not set, so the AI tutor is skipped.")

Reading the model's replies needs no key. This is how a reply becomes a translation and a
verdict (`True` = correct, `False` = incorrect, `None` = no verdict found):

In [ ]:
translationReply = "<think>...</think>\nTranslation: 「堅持」"
sentenceReply = "**VERDICT:** INCORRECT\nFEEDBACK: Say 'reluctant to go'."

print(genAITutor.cleanTranslation(translationReply))
print(genAITutor.parseSentenceFeedback(sentenceReply))

## 11. The whole app

`main()` is the menu you get from the `word-flashcard` command. It can run on the demo
deck too: set `RUN_APP = True`, run the cell, and type `q` to quit.

In [ ]:
RUN_APP = False

if RUN_APP:
    app.main(deckPath)

## Clean up

Delete the demo folder when you are done.

In [ ]:
shutil.rmtree(demoFolder, ignore_errors=True)
print("Removed", demoFolder)